# Working with Simulation Output


In this tutorial, we will take a closer look at what is in our simulation output. We will build upon our previous tutorial on using uproot and build the first steps of something we could use to analyse our data. Before we begin though, we will take a look at plotting our data as histograms, so that we can better understand what is going on. Of course, we now also have the benefit of a but of background information on what is actually going on in our detector and simulation. So now, we can begin to comprehend and interpret what our plots will show.

# Setup

Run the cells below once before running subsequent sections. The first one may take a minute or so. To summarise each cell -

- We are installing some packages (if they aren't already installed).
- Importing some packages.
- Setting a bunch of configuration options for our plots to make them look a bit nicer.

Once done, I recommend minimising this section.

In [ ]:
!pip install pandas
!pip install scipy

In [ ]:
#Import some packages we'll need, specifically, uproot
import uproot as up
import os
import awkward as ak
import numpy as np
import pandas as pd
import scipy
import matplotlib as mpl
import matplotlib.ticker as ticker
import matplotlib.cm as cm
import matplotlib.pylab as plt
from XRootD import client
from scipy import stats
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec

In [ ]:
plt.rcParams['figure.figsize'] = [8.0, 6.0]
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xaxis.labellocation'] = 'right'
plt.rcParams['yaxis.labellocation'] = 'top'
SMALL_SIZE = 10
MEDIUM_SIZE = 14
BIGGER_SIZE = 20
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title]
deg2rad = np.pi/180.0

# Basic Plotting Introduction

We will begin by taking a look at some of our reconstructed charged particles again.

In [ ]:
 # The file we downloaded previously, change as desired
fname = "/home/jovyan/eic/Day_1_Tutorial_Input.root"
if os.path.isfile(fname):
    file=up.open(fname)
else:
    print("Error opening file - ", fname, " check your fname variable!")

In [ ]:
tree = file['events']
ReconChPartBr = tree["ReconstructedChargedParticles"].arrays()

So long as we had no errors above, we should now have our file opened and our events tree assigned. We have then converted our ReconstructedChargedParticles branch elements to an array.

We can now go ahead and plot these as a histogram (relatively) straightforwardly.

**Note** - Feel free to rename variables as you like, I've just assigned these things names that make sense to *me*. This doesn't neccessarily mean they will make sense to you :)

In [ ]:
# Optional - remind yourself of the branches we have available by uncommeting the for loop below and running it
#for entry in ReconChPartBr.fields:
#    print(entry)
# Note that at this point, our ReconChPartBr variable is an AwkwardArray .fields provides a list of our branch names

## Making a Basic Histogram

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]))

As histograms go, not that informative. So what's gone wrong?

## Quick Quiz 

How can we get a better looking plot?

## Quick Quiz - Answer

We need to be a bit more careful in how our histogram is *binned*, we'll try setting it manually.

## Better Looking Histograms

To get a slightly more useful plot, let's try to set the number of bins (and binning range) ourselves. We can also supress that print out of numbers before the plot by adding "plt.show()"

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]), bins=100, range=(0,10))
plt.show()

Ok, better, but are we missing anything? Our plot before had a range going up to 1500.

This wasn't decided randomly, matplotlib tried to choose a range which would include all of our data. Maybe we should check some information regarding our array before deciding upon the binning.

In [ ]:
print("The smallest value in our array is:",np.min(ReconChPartBr["ReconstructedChargedParticles.energy"]))
print("The largest value in our array is:",np.max(ReconChPartBr["ReconstructedChargedParticles.energy"]))
print("The mean value of our array is:",np.mean(ReconChPartBr["ReconstructedChargedParticles.energy"]))
print("The standard deviation of values in our array is:",np.std(ReconChPartBr["ReconstructedChargedParticles.energy"]))

## Quick Quiz

Based upon these numbers, what might be a sensible range for our histogram?

## Quick Quiz - Answer/Discussion

Firstly, a quick comment here - There's no real *correct* answer to this. There are a few things to consider though -

- Whilst our max value is very high, the mean is a lot lower, so we probably don't want the range to extend up to the max value.
- Whatever range we pick, we probably don't want to bin too finely, overall, we don't have *that* many events here
    - This is a histogram, so we should carefully consider our bin range
- We don't *just* need to use information we printed from our array, we know something else about this data too
    - This is from a file with 18 GeV electrons colliding with 275 GeV protons.
    - So with this in mind, we probably wouldn't *really* expect energies in excess of a few hundred GeV
    - Let's try roughly 0.5 GeV per bin, covering a range from 0 up to 300 and see how that looks

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]), bins=600, range=(0,300))
plt.show()

Much better, but we're not actually using much of our range, we see most events between 0 and 25, with some slight fuzz above that. Let's reduce it to 0 to 50 (stil in 0.5 GeV bins)

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"]), bins=100, range=(0,50))
plt.show()

## Combining What We Know - Uproot Selection Masks/Cuts

So we can turn our arrays into histograms. Useful, but it still doesn't tell us that much. However, we can really start cooking when we combine this with what we ended with in our uproot session. We can apply selection cuts to our arrays to fiter out events we don't want. **Importantly**, we can apply cuts on **other** quantities as we draw the one we want. Let's use this to check the energy of our negatively charged reconstructed tracks.


In [ ]:
Positive = ReconChPartBr['ReconstructedChargedParticles.charge'] > 0
Negative = ReconChPartBr['ReconstructedChargedParticles.charge'] < 0
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50))
plt.show()

Nice, but how do our negative and positive charged particle energies compare? Well, we could plot them on top of each other and see -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive]), bins=100, range=(0,50),alpha=0.5,color='r')
plt.show()

We've added some extra options to our histograms here such that each plot has some transparency *and* the positively chagred particles are drawn in red rather than blue.

We could also have created the plot above slightly differently - 

In [ ]:
plt.hist([ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), 
          ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive])],
         bins=100, range=(0, 50),color=['b','r'])
# Note the square bracket enclosing our arrays and the colour selection we've chosen.
plt.show()

There are a few other tweaks we may wish to consider making to our histogram too.

### Histogram Plotting Options

There's a lot more we can (and probably should) do to our histogram to make it a bit nicer looking. To begin with, axis labels -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.show()

We can also add a title to our plot and set the x/y axis to be displayed logarithmically.

In this particular caplt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.show()se, that's maybe not so useful but, it's a option so let's try it -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5)
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.title("Energy of Negatively Charged Reconstructed Particles")
plt.xscale('log')
plt.yscale('log')
plt.show()

Not so useful, so we'll not do that again at the moment.

For our plot of positively and negatively charged particles on the same plot, we should also add a legend though so we know what we're looking at -

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5, label="-ve Particles")
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive]), bins=100, range=(0,50),alpha=0.5,color='r', label="+ve Particles")
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.title("Energy of Charged Reconstructed Particles")
plt.legend(loc='upper right')
plt.show()

Finally, we might want to save our histogram to a file.

In [ ]:
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Negative]), bins=100, range=(0,50),alpha=0.5, label="-ve Particles")
plt.hist(ak.flatten(ReconChPartBr["ReconstructedChargedParticles.energy"][Positive]), bins=100, range=(0,50),alpha=0.5,color='r', label="+ve Particles")
plt.xlabel('E [GeV]')
plt.ylabel('# Entries / 0.5 GeV')
plt.title("Energy of Charged Reconstructed Particles")
plt.legend(loc='upper right')
plt.savefig("MyFirstHistogram.png") # Note - the ordering matters here, if we do this AFTER showing the plot, the file will be blank
plt.show()

## Exercise

## Solution

# Basic Analysis